# Ficheiro 1 - Atividade de Internamento Hospitalar

# Exploração dos dados (ler CSV e explorar)

In [5]:

import pandas as pd

df = pd.read_csv("../data/raw/atividade-de-internamento-hospitalar_fich1.csv",
                  sep=";", encoding="utf-8-sig")

# Renomear colunas para snake_case sem acentos/espaços
df.columns = ["periodo", "regiao","instituicao","localizacao","especialidade","doentes_saidos","dias_internamento"]


print("Informações do DataFrame")
print("")

print("Dimensão do DataFrame (linhas, colunas):")
print(df.shape)          # (linhas, colunas)
print("")


print("Primeiras 5 linhas: ")
display(df.head())         # primeiras 5 linhas
print("")

print("Estatísticas básicas: ")
display(df.describe().round(2))     # estatísticas básicas das colunas numéricas
print("")

print("Valores em falta por coluna:")
print(df.isna().sum())   # quantos valores em falta por coluna
print("")

print("Verificar se há linhas duplicadas: ")
print(df.duplicated().sum())
print("")


print("Informações: ")
display(df.info())  # colunas, tipos e não nulos
print("")

print("Regiões:")
print(df["regiao"].unique())            # Ver que regiões são abordadas
print("")

print("Insituições:")
for inst in sorted(df["instituicao"].unique()):    # Ver que instituições são abordadas
    print(inst)


Informações do DataFrame

Dimensão do DataFrame (linhas, colunas):
(17616, 7)

Primeiras 5 linhas: 


,periodo,regiao,instituicao,localizacao,especialidade,doentes_saidos,dias_internamento
0,2015-05,Região de Saúde LVT,"Centro Hospitalar de Lisboa Ocidental, EPE","38.708454, -9.216985",Outras Camas,329.0,8327.0
1,2015-05,Região de Saúde Norte,"Centro Hospitalar Trás-os-Montes e Alto Douro,...","41.3031784, -7.7515252",Outras Camas,150.0,2392.0
2,2015-05,Região de Saúde Norte,"Hospital da Senhora da Oliveira, Guimarães, EPE","41.4387173, -8.3086907",Outras Camas,116.0,2066.0
3,2015-05,Região de Saúde Norte,"Hospital de Braga, PPP","41.56785, -8.398982",Outras Camas,71.0,1470.0
4,2015-05,Região de Saúde Norte,"Unidade Local de Saúde de Matosinhos, EPE","41.1794456, -8.6745115",Outras Camas,49.0,802.0



Estatísticas básicas: 


,doentes_saidos,dias_internamento
count,17588.00,17583.00
mean,3157.16,27088.15
std,3942.83,32915.38
min,0.00,37.00
25%,420.75,4929.00
50%,1710.00,15470.00
75%,4473.25,36047.50
max,34354.00,259313.00



Valores em falta por coluna:
periodo               0
regiao                0
instituicao           0
localizacao           0
especialidade         0
doentes_saidos       28
dias_internamento    33
dtype: int64

Verificar se há linhas duplicadas: 
0

Informações: 
<class 'pandas.DataFrame'>
RangeIndex: 17616 entries, 0 to 17615
Data columns (total 7 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   periodo            17616 non-null  str    
 1   regiao             17616 non-null  str    
 2   instituicao        17616 non-null  str    
 3   localizacao        17616 non-null  str    
 4   especialidade      17616 non-null  str    
 5   doentes_saidos     17588 non-null  float64
 6   dias_internamento  17583 non-null  float64
dtypes: float64(2), str(5)
memory usage: 963.5 KB


None


Regiões:
<StringArray>
[        'Região de Saúde LVT',       'Região de Saúde Norte',
   'Região de Saúde do Centro', 'Região de Saúde do Alentejo',
  'Região de Saúde do Algarve']
Length: 5, dtype: str

Insituições:
Centro Hospitalar Barreiro/Montijo, EPE
Centro Hospitalar Entre Douro e Vouga, EPE
Centro Hospitalar Médio Tejo, EPE
Centro Hospitalar Psiquiátrico de Lisboa
Centro Hospitalar Póvoa de Varzim/Vila do Conde, EPE
Centro Hospitalar Tondela-Viseu, EPE
Centro Hospitalar Trás-os-Montes e Alto Douro, EPE
Centro Hospitalar Tâmega e Sousa, EPE
Centro Hospitalar Universitário Cova da Beira, EPE
Centro Hospitalar Universitário Lisboa Central, EPE
Centro Hospitalar Universitário de Lisboa Norte, EPE
Centro Hospitalar Universitário de Santo António, EPE
Centro Hospitalar Universitário de São João, EPE
Centro Hospitalar Universitário do Algarve,EPE
Centro Hospitalar Universitário do Porto, EPE
Centro Hospitalar Vila Nova de Gaia/Espinho, EPE
Centro Hospitalar de Leiria, EPE
Centro Hosp

# Limpeza e Transformação dos dados
## Preparação do ficheiro para análise:

In [6]:
# Guardar o nº inicial de linhas, para comparar no fim:
ini = len(df)   

# Eliminar linhas onde ambos "doentes_saidos" e "dias_internamento" são nulos:
df = df[~(df["doentes_saidos"].isnull() & df["dias_internamento"].isnull())]

print("Eliminadas linhas onde ambos 'doentes_saidos' e 'dias_internamento' são nulos.")
print(f"Número linhas inicial: {ini}")
print("Depois:", len(df), "linhas")
print(f"Linhas eliminadas: {ini - len(df)}")
print("")

#-------------------------------------------------------------------------------------------------

# Separar coordenadas da coluna "localização_geográfica" em latitude e longitude
df[["latitude", "longitude"]] = df["localizacao"].str.split(",", expand=True)

# Converter para número
df["latitude"] = df["latitude"].astype(float)
df["longitude"] = df["longitude"].astype(float)

# Eliminar coluna original após separação
df = df.drop(columns=["localizacao"])

print("Latitude e longitude extraídas da coluna 'localizacao':")
print(df[["latitude", "longitude"]].head())
print("")

#-------------------------------------------------------------------------------------------------

# Uniformizar nomes regiões e instituições:

df["regiao"] = df["regiao"].replace(
    "Região de Saúde Norte",
    "Região de Saúde do Norte")


df["instituicao"] = df["instituicao"].replace(
    "EPE",
    "E.P.E.")


#-------------------------------------------------------------------------------------------------

# Calcular demora média apenas onde doentes_saidos > 0
df["demora_media"] = (df["dias_internamento"] / df["doentes_saidos"]).where(df["doentes_saidos"] > 0)

print("Demora média calculada (dias_internamento / doentes_saidos):")
print(df[["doentes_saidos", "dias_internamento", "demora_media"]].head())
print("")

#-------------------------------------------------------------------------------------------------

# Exportar ficheiro tratado
df.to_csv("../data/processed/atividade_de_internamento_tratado.csv", index=False)
print("Ficheiro exportado com sucesso!")


Eliminadas linhas onde ambos 'doentes_saidos' e 'dias_internamento' são nulos.
Número linhas inicial: 17616
Depois: 17604 linhas
Linhas eliminadas: 12

Latitude e longitude extraídas da coluna 'localizacao':
    latitude  longitude
0  38.708454  -9.216985
1  41.303178  -7.751525
2  41.438717  -8.308691
3  41.567850  -8.398982
4  41.179446  -8.674511

Demora média calculada (dias_internamento / doentes_saidos):
   doentes_saidos  dias_internamento  demora_media
0           329.0             8327.0     25.310030
1           150.0             2392.0     15.946667
2           116.0             2066.0     17.810345
3            71.0             1470.0     20.704225
4            49.0              802.0     16.367347

Ficheiro exportado com sucesso!
